# 🆘 AI Disaster Tweet Analyzer — Exploration Notebook

This notebook walks through the full crisis intelligence pipeline interactively:
1. Data generation / scraping
2. Text preprocessing
3. Model inference (BERT or rule-based fallback)
4. Hybrid location extraction
5. Priority classification
6. Result visualization

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Environment ready.')

## 1. Data Collection

In [ ]:
from utils.scraper import _generate_synthetic_tweets

raw_tweets = _generate_synthetic_tweets()
df = pd.DataFrame(raw_tweets)
print(f'Collected {len(df)} tweets')
df[['date', 'content', 'user_location', 'query_used']].head(5)

## 2. Preprocessing

In [ ]:
from utils.preprocessor import preprocess_dataframe

df = preprocess_dataframe(df)
df[['content', 'clean_text', 'keyword_score', 'hashtags']].head(5)

## 3. Model Inference

In [ ]:
from utils.inference import get_model_manager

manager = get_model_manager()
print(f'Model mode: {"BERT" if not manager.is_using_fallback else "Rule-based fallback"}')

# Test a few examples
test_texts = [
    'URGENT: We are trapped on the roof! Flood water rising fast! Send rescue NOW!',
    'Red Cross shelter opened at Lincoln High School. Free meals available.',
    'Entire neighborhood underwater. Hundreds of homes destroyed by flood.',
]
for text in test_texts:
    result = manager.predict_single(text)
    print(f'\nText: {text[:60]}...')
    print(f'  Category: {result["category"]}')
    print(f'  Urgency:  {result["urgency_score"]:.1f}')
    print(f'  Probs:    {result["all_probs"]}')

## 4. Batch Inference on Dataset

In [ ]:
predictions = manager.predict_batch(df['clean_text'].tolist())
df['category'] = [p['category'] for p in predictions]
df['urgency_score'] = [p['urgency_score'] for p in predictions]

print('Category distribution:')
print(df['category'].value_counts())
print(f'\nUrgency score stats:')
print(df['urgency_score'].describe())

## 5. Hybrid Location Extraction

In [ ]:
from utils.location_extractor import extract_location, get_priority_label, load_spacy_model

nlp = load_spacy_model()

# Demo extraction
demo_cases = [
    {'tweet_text': 'URGENT flood in Houston, TX! Rescue needed NOW! #HoustonFlood',
     'hashtags_str': 'HoustonFlood', 'profile_location': 'Houston, TX', 'user_bio': ''},
    {'tweet_text': 'Earthquake just hit hard. Buildings falling. Send help!',
     'hashtags_str': 'LAEarthquake', 'profile_location': '', 'user_bio': 'LA native'},
    {'tweet_text': 'We need rescue. Trapped. No water.',
     'hashtags_str': '', 'profile_location': '', 'user_bio': 'father of three'},
]
for case in demo_cases:
    result = extract_location(**case, nlp=nlp)
    print(f'Tweet: {case["tweet_text"][:55]}...')
    print(f'  → {result}\n')

## 6. Full Enrichment + Priority Scoring

In [ ]:
locations, sources, confidences, priorities = [], [], [], []

for _, row in df.iterrows():
    loc = extract_location(
        tweet_text=row.get('content', ''),
        hashtags_str=row.get('hashtags_str', ''),
        profile_location=row.get('user_location', ''),
        user_bio=row.get('user_bio', ''),
        nlp=nlp,
    )
    locations.append(loc['location'])
    sources.append(loc['location_source'])
    confidences.append(loc['location_confidence'])
    priorities.append(get_priority_label(row['urgency_score'], loc['location_source']))

df['location'] = locations
df['location_source'] = sources
df['location_confidence'] = confidences
df['priority'] = priorities

print('Priority distribution:')
print(df['priority'].value_counts())
print(f'\nLocation source distribution:')
print(df['location_source'].value_counts())

## 7. Top Urgent Tweets

In [ ]:
output_cols = ['content', 'category', 'urgency_score', 'location', 'location_source', 'location_confidence', 'priority']
top = df.sort_values('urgency_score', ascending=False)[output_cols].head(10)

pd.set_option('display.max_colwidth', 60)
top

## 8. Critical Review Flag — High Urgency + Unknown Location

In [ ]:
critical = df[(df['urgency_score'] > 70) & (df['location'] == 'unknown')]
print(f'Critical review tweets (urgent + no location): {len(critical)}')
critical[['content', 'urgency_score', 'category']].head()